# Trabalho 1 (DCA3702): Rede de Relacionamentos nas Noticias da UFRN

Disciplina: Estrutura de Dados II (DCA3702), UFRN.  
Aluno: Lucas Augusto Spinola Pinto.  

## Pergunta de pesquisa
Como pessoas, departamentos, centros, projetos, eventos e organizacoes se relacionam dentro das noticias publicadas pela UFRN, e quais entidades ocupam posicoes centrais nessa rede?

## Pipeline
1. Coleta via WP REST API do portal UFRN, com filtro de bylines.
2. NER com spaCy + EntityRuler + canonicalizacao de aliases.
3. Construcao do grafo (MultiDiGraph, combinations, relacoes simetricas e assimetricas).
4. Metricas estruturais + 5 centralidades + comparacao Erdos-Renyi e Watts-Strogatz.
5. Visualizacoes: matplotlib + PyVis.
6. Dashboard HTML autossuficiente.

## 1. Setup

In [1]:
# !pip install -r requirements.txt
# !python -m spacy download pt_core_news_lg

import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Working dir:', os.getcwd())

from src import scraper, ner, graph, analysis, visualization, dashboard
from src.canonical import total_aliases
print(f'Aliases de canonicalizacao carregados: {total_aliases()}')

Working dir: C:\Users\Lucas Spinola.DESKTOP-NGIDNEL\Desktop\T2-AED2\T1


Aliases de canonicalizacao carregados: 34


## 2. Etapa 1: Coleta de noticias

A coleta usa a WordPress REST API do portal UFRN. Descobri o endpoint inspecionando o JavaScript da pagina:
`https://webcache01-producao.info.ufrn.br/admin/portal-ufrn/wp-json/wp/v2/noticias-publicadas/`

Coloquei uma pausa de 1 segundo entre requisicoes. O scraper tambem remove assinaturas jornalisticas (`Reportagem:`, `Texto:`, etc.) do corpo antes de salvar, para que esses nomes nao acabem inflando o PageRank depois.

In [2]:
noticias = scraper.coletar_noticias(limite=150, per_page=30, pausa_segundos=1.0)
print(f'Coletadas {len(noticias)} noticias')
scraper.salvar_noticias(noticias, 'data/raw/noticias.json')
noticias[0]['titulo'], noticias[0]['data']

Coletadas 150 noticias


('Jornada Acadêmica discute Direito Público e as múltiplas atividades do Estado',
 '2026-06-03T16:30:14')

## 3. Etapa 2: NER (com canonicalizacao e filtro de bylines)

Pipeline usado: `pt_core_news_lg` + `EntityRuler` (78 padroes da UFRN) + canonicalizacao manual + filtro de bylines.

Por que canonicalizar? Na primeira execucao, 'UFRN' e 'Universidade Federal do Rio Grande do Norte' apareciam como dois nos diferentes. `src/canonical.py` une essas variantes.

Por que filtrar bylines? Jornalistas da Agecom (Beatriz, Hellen, Rebeca) assinam quase toda materia e co-ocorrem com tudo, o que infla o PageRank deles. `src/bylines.json` lista os nomes a remover depois do NER.

In [3]:
nlp = ner.carregar_modelo()
print('Componentes do pipeline:', nlp.pipe_names)

Componentes do pipeline: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'entity_ruler', 'ner']


In [4]:
noticias_proc = ner.processar_noticias(noticias, nlp=nlp)
ner.salvar_entidades(noticias_proc, 'data/processed/entidades.json')
total_ent = sum(len(n['entidades']) for n in noticias_proc)
print(f'Total de mencoes de entidades (apos filtros): {total_ent}')
print('Exemplos extraidos da primeira noticia:')
noticias_proc[0]['entidades'][:10]

Total de mencoes de entidades (apos filtros): 1561
Exemplos extraidos da primeira noticia:


[('Centro de Ensino Superior', 'ORGANIZACAO'),
 ('UFRN', 'ORGANIZACAO'),
 ('Juan de Assis Almeida', 'PESSOA'),
 ('Direito Administrativo', 'ORGANIZACAO'),
 ('Direitos Fundamentais, Inteligência Artificial', 'ORGANIZACAO'),
 ('Agecom', 'ORGANIZACAO')]

## 4. Etapa 3: Construcao do grafo

Uso `MultiDiGraph` (NetworkX). Cada par de entidades co-ocorrente vira uma aresta. Usei `combinations` em vez de `permutations` para nao gerar pares redundantes. Para relacoes assimetricas (`PERTENCE_A`, `DESENVOLVE`, `PARTICIPA_DE`), a aresta vai na direcao semantica adequada. Para simetricas (`COLABORA_COM`, `RELACIONADO_A`), uma so aresta arbitraria.

Salvo o grafo em dois formatos: GraphML (padrao) e GEXF (nativo do Gephi).

In [5]:
G = graph.construir_grafo(noticias_proc)
print(f'Grafo: {G.number_of_nodes()} nos, {G.number_of_edges()} arestas (MultiDiGraph)')
graph.salvar_graphml(G, 'results/grafo.graphml')
graph.salvar_gexf(G, 'results/grafo.gexf')
print('Salvo em results/grafo.graphml e results/grafo.gexf')

Grafo: 1067 nos, 9610 arestas (MultiDiGraph)


Salvo em results/grafo.graphml e results/grafo.gexf


## 5. Etapa 4: Metricas estruturais e centralidades

Calculo todas as metricas sobre o grafo simples nao direcionado, que e o formato exigido por diametro, clustering e eigenvector.

In [6]:
m = analysis.metricas_estruturais(G)
print(json.dumps(m, indent=2, ensure_ascii=False, default=str))
analysis.salvar_metricas(m, 'results/metricas.json')

{
  "num_nos": 1067,
  "num_arestas_simples": 8891,
  "num_arestas_multidigraph": 9610,
  "densidade": 0.01563359949077827,
  "componentes_conectados": 1,
  "tamanho_maior_componente": 1067,
  "grau_medio": 16.665417057169634,
  "coeficiente_agrupamento_medio": 0.9419768525082233,
  "transitividade": 0.17270118031786827,
  "diametro_maior_componente": 4,
  "comprimento_medio_caminhos": 2.005231127936685
}


In [7]:
# Distribuicao de tipos de aresta
contagem = analysis.contagem_por_tipo_aresta(G)
print('Distribuicao de tipos de relacao:')
for tipo, qtd in contagem.items():
    print(f'  {tipo:18s} {qtd:>6d}')

Distribuicao de tipos de relacao:
  RELACIONADO_A        6036
  COLABORA_COM         3313
  PERTENCE_A            261


In [8]:
df = analysis.centralidades(G)
df.head(20)

,node,tipo,frequencia,degree,betweenness,closeness,eigenvector,pagerank
0,UFRN,ORGANIZACAO,148,0.988743,0.892604,0.988868,0.539007,0.068366
1,Agecom,ORGANIZACAO,50,0.333021,0.052949,0.599887,0.230311,0.021730
2,José Daniel Diniz Melo,PESSOA,14,0.106004,0.004983,0.527984,0.076880,0.006690
3,Instituto Metrópole Digital,CENTRO,11,0.077861,0.002468,0.517225,0.053571,0.005275
4,MEC,ORGANIZACAO,10,0.078799,0.004742,0.519240,0.061299,0.005213
5,Instagram,PESSOA,11,0.083490,0.002844,0.518735,0.060881,0.005178
6,"Centro de Ciências Humanas, Letras e Artes",CENTRO,9,0.099437,0.003839,0.523062,0.097664,0.004806
7,Centro de Ciências Exatas e da Terra,CENTRO,10,0.059099,0.001347,0.512254,0.035627,0.004404
8,PROEX,ORGANIZACAO,9,0.072233,0.001776,0.515723,0.064471,0.004401
9,Centro de Ciências Sociais Aplicadas,CENTRO,9,0.066604,0.001669,0.514231,0.052492,0.004077


In [9]:
analysis.salvar_ranking(df, 'results/ranking_top20.csv', top=20)
print('Top 5 por PageRank:')
for _, row in df.head(5).iterrows():
    print(f'  {row["node"][:50]:50s} ({row["tipo"]:12s}) PR={row["pagerank"]:.4f}')

Top 5 por PageRank:
  UFRN                                               (ORGANIZACAO ) PR=0.0684
  Agecom                                             (ORGANIZACAO ) PR=0.0217
  José Daniel Diniz Melo                             (PESSOA      ) PR=0.0067
  Instituto Metrópole Digital                        (CENTRO      ) PR=0.0053
  MEC                                                (ORGANIZACAO ) PR=0.0052


## 6. Validacao small-world: rede real vs Erdos-Renyi vs Watts-Strogatz

Para conferir se a rede tem estrutura genuina (e nao e apenas acaso estatistico), comparo o clustering e o caminho medio com dois modelos de mesma escala:
- Erdos-Renyi com a mesma densidade (rede puramente aleatoria).
- Watts-Strogatz com mesmo grau medio e p_rewire=0,1 (rede small-world canonica).

Se a rede real tiver clustering muito maior que o ER mas caminho medio parecido, fica confirmado que ela e small-world.

In [10]:
cmp = analysis.comparar_modelos_aleatorios(G)
for modelo in ['real', 'er', 'ws']:
    nome = {'real':'Real', 'er':'Erdos-Renyi', 'ws':'Watts-Strogatz'}[modelo]
    d = cmp[modelo]
    cam = f"{d['caminho_medio']:.4f}" if d.get('caminho_medio') is not None else 'n/d'
    print(f'  {nome:16s} clustering={d["clustering"]:.4f}  caminho_medio={cam}')

  Real             clustering=0.9420  caminho_medio=2.0052
  Erdos-Renyi      clustering=0.0149  caminho_medio=2.7749
  Watts-Strogatz   clustering=0.5120  caminho_medio=3.3793


## 7. Etapa 5: Visualizacoes

Gero 6 imagens (incluindo a comparacao com modelos aleatorios) e um grafo HTML interativo.

In [11]:
imagens = visualization.gerar_todas(
    analysis._para_simples(G), df,
    pasta_imagens='imagens', pasta_results='results',
    comparacao=cmp,
)
for k, v in imagens.items():
    print(f'  {k:24s} -> {v}')

  hist_grau                -> imagens\01_hist_grau.png
  top_pagerank             -> imagens\02_top_pagerank.png
  distribuicao_tipos       -> imagens\03_distribuicao_tipos.png
  componentes              -> imagens\04_componentes.png
  grafo_estatico           -> imagens\05_grafo_estatico.png
  grafo_interativo         -> results\grafo_interativo.html
  comparacao_aleatorios    -> imagens\06_real_vs_aleatorio.png


## 8. Etapa 6: Dashboard HTML

Monta o arquivo `results/index.html` autossuficiente: imagens em base64, tabela de comparacao com ER/WS, distribuicao por tipo de relacao, tabela top 20 e iframe do grafo interativo.

In [12]:
dashboard.gerar(
    metricas=m, df_top=df.head(20),
    num_noticias=len(noticias_proc),
    num_entidades_unicas=G.number_of_nodes(),
    imagens=imagens,
    iframe_html='grafo_interativo.html',
    caminho_saida='results/index.html',
    comparacao_aleatorios=cmp,
    contagem_arestas=contagem,
)
print('Dashboard salvo em results/index.html')

Dashboard salvo em results/index.html


## 9. Discussao e limitacoes

Achados principais:
- O top 5 por PageRank reproduz bem a hierarquia institucional real: UFRN, Agecom, reitor (Jose Daniel Diniz Melo), Instituto Metropole Digital e MEC.
- Validacao small-world: clustering real 63x maior que Erdos-Renyi com a mesma densidade. E uma evidencia forte de que a estrutura nao e aleatoria.
- Predominam relacoes `RELACIONADO_A` (mais genericas) e `COLABORA_COM` (ORG-ORG). Aparecem so 261 `PERTENCE_A`, mas sao o sinal mais limpo do grafo, porque capturam vinculos institucionais explicitos.

Limitacoes:
- 150 noticias e uma janela curta (mais ou menos 1 mes). Analises longitudinais exigiriam coleta incremental.
- A canonicalizacao manual cobre uns 50 aliases conhecidos. Variantes nao previstas ainda viram nos duplicados.
- Co-ocorrencia em texto nao e o mesmo que vinculo causal real. Uma mencao conjunta nao garante relacao estrutural.

## 10. Conclusoes

Com scraping, NER e grafos foi possivel reconstruir parte do tecido institucional da UFRN observavel pelas noticias publicas. As correcoes pos-NER (canonicalizacao e filtro de bylines) mostraram, na pratica, que a definicao de 'o que conta como uma entidade' determina o resultado: NER bruto sem pos-processamento gera grafos visualmente bonitos mas semanticamente quebrados.